In [1]:
import pandas as pd
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [2]:
import tensorflow as tf
devices = tf.config.list_physical_devices('GPU')
if devices:
    print("GPU found!")

Metal GPU Bulundu!


In [3]:
import tensorflow as tf

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU Enabled: {gpu_devices}")
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("GPU could not found, Tasks will run on CPU")

M4 GPU Aktif: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
df = pd.read_csv("data/spam_Emails_data.csv")
df.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [7]:
texts = df['text'].values
labels = df['label'].map({'Ham': 0, 'Spam': 1}).values # Manuel eşleme daha güvenlidir

In [8]:
import pandas as pd

df = df.dropna(subset=['text'])
texts = df['text'].astype(str).values


In [9]:
# 2. Tokenization Encoding words.
max_words = 20000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

In [10]:
max_len = 150
X = pad_sequences(sequences, maxlen=max_len)

In [13]:
from tensorflow.keras.layers import Bidirectional

model = Sequential([

    Embedding(input_dim=max_words, output_dim=128),

    # 2. Bidirectional LSTM: Reads text from start to end, end to start.
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.25),

    LSTM(64),
    Dropout(0.25),


    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [14]:
# Train the model in 6 epoch.
model.fit(X, labels, epochs=6,batch_size=1024, validation_split=0.2) # M4 GPU allows 1024 batch size.

Epoch 1/10


2026-03-17 09:21:26.179188: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


152/152 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.9307 - loss: 0.1699 - val_accuracy: 0.9759 - val_loss: 0.0699
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 277s 2s/step - accuracy: 0.9833 - loss: 0.0532 - val_accuracy: 0.9838 - val_loss: 0.0508
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 250s 2s/step - accuracy: 0.9897 - loss: 0.0334 - val_accuracy: 0.9852 - val_loss: 0.0475
Epoch 4/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.9934 - loss: 0.0217 - val_accuracy: 0.9810 - val_loss: 0.0588
Epoch 5/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 257s 2s/step - accuracy: 0.9951 - loss: 0.0167 - val_accuracy: 0.9874 - val_loss: 0.0456
Epoch 6/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 259s 2s/step - accuracy: 0.9974 - loss: 0.0097 - val_accuracy: 0.9842 - val_loss: 0.0600
Epoch 7/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.9969 - loss: 0.0106 - val_accuracy: 0.9826 - val_loss: 0.0647
Epoch 8/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.9975 - loss: 0.0092 - val_accuracy: 0.987

In [15]:
import numpy as np

# Random texts for testing mail.
random_mails = [
    "LOOK HERE YOU SON OF A *****! YOU WIN 1000$ FREE SPIN COUPON.",
    "BEST BET SITE HERE, YOU WON 100$ FREE SPIN COUPON",
    "See you in the lab. Do not forget to bring your circuit"
]

# Tokenize mails
new_seq = tokenizer.texts_to_sequences(random_mails)
new_pad = pad_sequences(new_seq, maxlen=max_len)

# Prediction
preds = model.predict(new_pad)

# Results
for i, mail in enumerate(random_mails):
    prob = preds[i][0]
    state = ("KATIKSIZ SPAM" if prob > 0.75 else "SPAM") if prob > 0.5 else "SECURE"
    print(f"\nMail: {mail}")
    print(f"{state} (Probability: %{prob*100:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step

Mail: LOOK HERE YOU SON OF A BITCH! YOU WIN 1000$ FREE SPIN COUPON.
Tahmin: KATIKSIZ SPAM (Olasılık: %99.92)

Mail: BEST BET SITE HERE, YOU WON 100$ FREE SPIN COUPON
Tahmin: KATIKSIZ SPAM (Olasılık: %99.96)

Mail: Sarp, see you in the lab. Do not forget to bring your circuit
Tahmin: SECURE (Olasılık: %0.01)


In [16]:
# Save model to a file
model.save('spam_model_v2.h5')

In [17]:
import pickle
with open('tokenizer2.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)